In [17]:
import pandas as pd
import numpy as np
import json
import gzip
from tqdm import tqdm
import os
import ast

META_MOVIE_PATH = "../meta_Movies_and_TV.json.gz"
META_BOOK_PATH = "../meta_Books.json.gz"
SAVE_DIR = "./movie_book_cdsr_processed/"
VALID_ITEM_SET_PATH = "./movie_book_cdsr_processed/valid_item_set.json"
ITEM2ID_PATH = "./movie_book_cdsr_processed/item2id.json"
os.makedirs(SAVE_DIR, exist_ok=True)

In [18]:
# 加载meta数据，只保留valid_item_set中的商品
def load_and_filter_meta(file_path, domain_name, valid_item_set):
    print(f"正在处理 {domain_name} 域元数据...")
    meta_list = []
    
    with gzip.open(file_path, 'rb') as f:    
        for line in tqdm(f, desc=f"读取{domain_name} meta"):
            line_str = line.decode('latin-1').strip()
            item = ast.literal_eval(line_str)  
            
            asin = item.get("asin", "")
            # 只处理有效商品集合里的商品
            if asin not in valid_item_set:
                continue
            # 标题、描述、图片URL
            title = item.get("title", "").strip()
            if not title:
                continue
                
            desc = item.get("description", "")
            desc = desc.strip() if isinstance(desc, str) else ""
            imUrl = item.get("imUrl", "").strip()
            
            meta_list.append({
                "item_id": asin,
                "domain": domain_name,
                "title": title,
                "description": desc,
                "imUrl": imUrl
            })
    # 转为DataFrame
    df = pd.DataFrame(meta_list).drop_duplicates(subset=["item_id"]).reset_index(drop=True)
    print(f"{domain_name} 域元数据处理完成，匹配到商品数：{len(df)}")
    return df

In [19]:
with open(VALID_ITEM_SET_PATH, "r", encoding="utf-8") as f:
    valid_item_set = set(json.load(f))
with open(ITEM2ID_PATH, "r", encoding="utf-8") as f:
    item2id = json.load(f)
with open(f"{SAVE_DIR}/domain2id.json", "r", encoding="utf-8") as f:
    domain2id = json.load(f)

print(f"有效商品数：{len(valid_item_set)}")
# 加载并过滤两个域的元数据
print("加载并过滤元数据...")
movie_meta = load_and_filter_meta(META_MOVIE_PATH, "movie", valid_item_set)
book_meta = load_and_filter_meta(META_BOOK_PATH, "book", valid_item_set)
full_meta = pd.concat([movie_meta, book_meta], ignore_index=True)

# 映射item_id和domain到整数ID
full_meta["item_id"] = full_meta["item_id"].map(item2id)
full_meta["domain"] = full_meta["domain"].map(domain2id)

# 按item_id排序
full_meta = full_meta.sort_values(by="item_id").reset_index(drop=True)

# 保存
output_path = f"{SAVE_DIR}/item_meta.csv"
full_meta.to_csv(output_path, index=False, encoding="utf-8")

print(f"最终匹配到的有效商品数：{len(full_meta)}")
print(f"文件已保存至：{output_path}")

有效商品数：175916
加载并过滤元数据...
正在处理 movie 域元数据...


读取movie meta: 208321it [00:15, 13465.75it/s]


movie 域元数据处理完成，匹配到商品数：13271
正在处理 book 域元数据...


读取book meta: 2370585it [02:27, 16085.55it/s]


book 域元数据处理完成，匹配到商品数：114781
最终匹配到的有效商品数：128052
文件已保存至：./movie_book_cdsr_processed//item_meta.csv
